# Laboratorio 6: Bases de Datos con Python — SQLite y MongoDB

Bienvenido al Laboratorio 6. En esta sesión exploraremos las dos caras de la moneda del almacenamiento de datos: el mundo **Relacional (SQL)** con SQLite y el mundo **No Relacional (NoSQL)** con MongoDB. Aprenderás a modelar datos, garantizar la integridad referencial y manipular documentos de forma eficiente desde Python.

## 📋 Tabla de Contenidos

1. [PARTE 1: SQLite desde la Terminal](#-sqlite-terminal)
2. [PARTE 2: Modelo Relacional (User-Post-Comment)](#-modelo-relacional)
3. [PARTE 3: SQLModel en Python](#-sqlmodel)
4. [PARTE 4: Operaciones CRUD con SQLModel](#-crud-sql)
5. [PARTE 5: MongoDB desde Mongo Shell](#-mongo-shell)
6. [PARTE 6: PyMongo en Python](#-pymongo)
7. [PARTE 7: MongoEngine (ODM)](#-mongoengine)
8. [PARTE 8: Comparativa: SQL vs NoSQL](#-comparativa)

---

<a id='-modelo-relacional'></a>
## 🔹 2: PARTE 2: Modelo Relacional (User-Post-Comment)

En una base de datos relacional, los datos se organizan en tablas conectadas por relaciones. Usaremos el modelo **User -> Post -> Comment** para entender la **Integridad Referencial**.

| Tabla | Campos Principales | Relación |
| :--- | :--- | :--- |
| **User** | `id (PK)`, `username`, `married (bool)`, `dates` | Dueño de los Posts. |
| **Post** | `id (PK)`, `title`, `content`, `user_id (FK)` | Pertenece a 1 Usuario (1:M). |
| **Comment**| `id (PK)`, `content`, `post_id (FK)`, `user_id (FK)` | Conecta Usuario y Post. |

---

<a id='-sqlite-terminal'></a>
## 🔹 1: PARTE 1: SQLite desde la Terminal

SQLite es un motor de base de datos ligero que **no requiere un servidor**. Todo se almacena en un simple archivo local. Antes de usar Python, es vital entender cómo funciona desde la línea de comandos.

### Conceptos Clave:
- **Crear DB:** `sqlite3 mi_base.db` en la terminal.
- **Ver Esquema:** `.schema` muestra el código SQL de tus tablas.
- **Ver bonito:** `.mode column && .headers on`
- **Comandos Básicos:**
  - `CREATE TABLE`: Define la estructura.
  - `INSERT`: Agrega datos.
  - `SELECT`: Consulta datos.

```sql
-- 1. Definición del esquema (Tablas y Relaciones)
CREATE TABLE IF NOT EXISTS user (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT NOT NULL UNIQUE,
    married BOOLEAN DEFAULT 0,
    created_at DATETIME DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS post (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    content TEXT NOT NULL,
    user_id INTEGER NOT NULL,
    created_at DATETIME DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (user_id) REFERENCES user (id)
);

CREATE TABLE IF NOT EXISTS comment (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    content TEXT NOT NULL,
    user_id INTEGER NOT NULL,
    post_id INTEGER NOT NULL,
    created_at DATETIME DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (user_id) REFERENCES user (id),
    FOREIGN KEY (post_id) REFERENCES post (id)
);

-- 2. CRUD
-- INSERT
INSERT INTO user (username, married) VALUES ('jorge_laco', 0);
INSERT INTO user (username, married) VALUES ('maria_db', 1);

INSERT INTO post (title, content, user_id) VALUES ('Mi primer post', 'Contenido sobre Cloud Architecture', 1);
INSERT INTO post (title, content, user_id) VALUES ('SQLite Tips', 'Usando llaves foráneas eficientemente', 2);

INSERT INTO comment (content, user_id, post_id) VALUES ('Excelente post!', 2, 1);
INSERT INTO comment (content, user_id, post_id) VALUES ('Me sirvió mucho, gracias.', 1, 2);

-- READ
SELECT * FROM user;
SELECT * FROM post;
SELECT * FROM comment;

-- ACTUALIZAR
UPDATE user 
SET married = 1 
WHERE username = 'jorge_laco';

UPDATE post 
SET content = 'Contenido actualizado sobre Cloud Architecture en AWS' 
WHERE id = 1;

UPDATE comment 
SET content = 'Excelente post! Muy recomendado.' 
WHERE id = 1;

-- DELETE
DELETE FROM comment WHERE id = 2;

-- CONSULTA UNIFICADA (One query to rule them all)
SELECT 
    u.id AS user_id,
    u.username,
    u.married,
    p.id AS post_id,
    p.title AS post_title,
    p.content AS post_content,
    c.id AS comment_id,
    c.content AS comment_text,
    c.created_at AS comment_date
FROM user u
LEFT JOIN post p ON u.id = p.user_id
LEFT JOIN comment c ON p.id = c.post_id;
```

---

<a id='-sqlmodel'></a>
## 🔹 3: PARTE 3: SQLModel en Python

**SQLModel** es la librería moderna de Python que combina el poder de SQLAlchemy con la simplicidad de Pydantic. Es la forma más limpia de manejar SQL hoy en día.

In [54]:
# uv add sqlmodel

In [1]:
from datetime import datetime
from sqlmodel import SQLModel, Field, Relationship, create_engine, Session, select, text
from sqlalchemy.exc import IntegrityError

# 1. Definición de Modelos
class User(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    username: str = Field(index=True, unique=True)
    first_name: str
    last_name: str
    married: bool = False
    created_at: datetime = Field(default_factory=datetime.now)
    
    # Relaciones
    posts: list["Post"] = Relationship(back_populates="author")
    comments: list["Comment"] = Relationship(back_populates="author")

class Post(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    title: str
    content: str
    user_id: int = Field(foreign_key="user.id", ondelete="CASCADE")
    created_at: datetime = Field(default_factory=datetime.now)
    
    # Relaciones
    author: User = Relationship(back_populates="posts")
    comments: list["Comment"] = Relationship(back_populates="post", cascade_delete=True)

class Comment(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    content: str
    user_id: int = Field(foreign_key="user.id", ondelete="CASCADE")
    post_id: int = Field(foreign_key="post.id", ondelete="CASCADE")
    created_at: datetime = Field(default_factory=datetime.now)
    
    # Relaciones
    author: User = Relationship(back_populates="comments")
    post: Post = Relationship(back_populates="comments")

# 2. Configuración del Engine (SQLite Local)
sqlite_file = "blog.db"
sqlite_url = f"sqlite:///{sqlite_file}"
engine = create_engine(sqlite_url, echo=False)

def create_db_and_tables():
    SQLModel.metadata.create_all(engine)
    print("✅ Base de datos y tablas creadas exitosamente.")

In [2]:
create_db_and_tables()

✅ Base de datos y tablas creadas exitosamente.


---

<a id='-crud-sql'></a>
## 🔹 4: PARTE 4: Operaciones CRUD Completas

Aprenderemos a crear, leer, actualizar y eliminar registros usando funciones limpias.

In [3]:
def create_data(users: list[User], posts: list[Post], comments: list[Comment]):
    with Session(engine) as session:
        # Prevent duplicates by checking if the table already has data
        if session.exec(select(User)).first():
            print("⚠️ Initial data already exists. Skipping creation.")
            return

        # Add all the passed objects to the session
        session.add_all(users)
        session.add_all(posts)
        session.add_all(comments)
        
        session.commit()
        print("✅ CREATE: Dynamic data successfully inserted.")

In [4]:
# 1. Define the instances in memory
u1 = User(username="jorge_laco", first_name="Jorge", last_name="Laco", married=False)
u2 = User(username="maria_db", first_name="Maria", last_name="DB", married=True)

# Link posts to authors using the python object
p1 = Post(title="Mi primer post", content="Contenido sobre Cloud Architecture", author=u1)
p2 = Post(title="SQLite Tips", content="Usando llaves foráneas eficientemente", author=u2)

# Link comments to authors and posts
c1 = Comment(content="Excelente post!", author=u2, post=p1)
c2 = Comment(content="Me sirvió mucho, gracias.", author=u1, post=p2)

# 2. Group them into lists
lista_usuarios = [u1, u2]
lista_posts = [p1, p2]
lista_comentarios = [c1, c2]

# 3. Pass them into your clean function
create_data(users=lista_usuarios, posts=lista_posts, comments=lista_comentarios)

✅ CREATE: Dynamic data successfully inserted.


In [5]:
VALID_TABLES = {"user", "post", "comment"}

def read_all_raw(table_name):
    table_name = table_name.lower()
    if table_name not in VALID_TABLES:
        raise ValueError(f"Tabla inválida: {table_name}. Permisibles: {VALID_TABLES}")
    query = f"SELECT * FROM {table_name}"
    
    with Session(engine) as session:
        result = session.execute(text(query))
        for row in result.mappings():
            print(row)

In [6]:
read_all_raw("user")
read_all_raw("post")
read_all_raw("comment")

{'id': 1, 'username': 'jorge_laco', 'first_name': 'Jorge', 'last_name': 'Laco', 'married': 0, 'created_at': '2026-06-04 14:59:24.759962'}
{'id': 2, 'username': 'maria_db', 'first_name': 'Maria', 'last_name': 'DB', 'married': 1, 'created_at': '2026-06-04 14:59:24.761222'}
{'id': 1, 'title': 'Mi primer post', 'content': 'Contenido sobre Cloud Architecture', 'user_id': 1, 'created_at': '2026-06-04 14:59:24.762133'}
{'id': 2, 'title': 'SQLite Tips', 'content': 'Usando llaves foráneas eficientemente', 'user_id': 2, 'created_at': '2026-06-04 14:59:24.763243'}
{'id': 1, 'content': 'Excelente post!', 'user_id': 2, 'post_id': 1, 'created_at': '2026-06-04 14:59:24.764293'}
{'id': 2, 'content': 'Me sirvió mucho, gracias.', 'user_id': 1, 'post_id': 2, 'created_at': '2026-06-04 14:59:24.765346'}


In [7]:
def read_one_raw(table_name, record_id):
    query = f"SELECT * FROM {table_name} WHERE id = :id"
    
    with Session(engine) as session:
        result = session.execute(text(query), {"id": record_id})
        row = result.mappings().first()
        print(row)

In [8]:
read_one_raw("user", record_id=1)
read_one_raw("post", record_id=1)
read_one_raw("comment", record_id=1)

{'id': 1, 'username': 'jorge_laco', 'first_name': 'Jorge', 'last_name': 'Laco', 'married': 0, 'created_at': '2026-06-04 14:59:24.759962'}
{'id': 1, 'title': 'Mi primer post', 'content': 'Contenido sobre Cloud Architecture', 'user_id': 1, 'created_at': '2026-06-04 14:59:24.762133'}
{'id': 1, 'content': 'Excelente post!', 'user_id': 2, 'post_id': 1, 'created_at': '2026-06-04 14:59:24.764293'}


In [9]:
VALID_COLUMNS = {"username", "first_name", "last_name", "married", "title", "content"}

def update_record_raw(table_name, record_id, column_name, new_value):
    table_name = table_name.lower()
    column_name = column_name.lower()
    if table_name not in VALID_TABLES:
        raise ValueError(f"Tabla inválida: {table_name}. Permisibles: {VALID_TABLES}")
    if column_name not in VALID_COLUMNS:
        raise ValueError(f"Columna inválida: {column_name}. Permisibles: {VALID_COLUMNS}")
    query = f"UPDATE {table_name} SET {column_name} = :new_value WHERE id = :id"
    
    with Session(engine) as session:
        session.execute(text(query), {"new_value": new_value, "id": record_id})
        session.commit()
        return True

In [10]:
# 1. Update a user's married status
update_record_raw(
    table_name="user", 
    record_id=1, 
    column_name="married", 
    new_value=True
)

# 2. Edit a post's content
update_record_raw(
    table_name="post", 
    record_id=1, 
    column_name="content", 
    new_value="Contenido actualizado sobre Cloud Architecture en AWS"
)

# 3. Modify a specific comment
update_record_raw(
    table_name="comment", 
    record_id=1, 
    column_name="content", 
    new_value="Excelente post! Muy recomendado."
)

True

In [11]:
read_all_raw("user")
read_all_raw("post")
read_all_raw("comment")

{'id': 1, 'username': 'jorge_laco', 'first_name': 'Jorge', 'last_name': 'Laco', 'married': 1, 'created_at': '2026-06-04 14:59:24.759962'}
{'id': 2, 'username': 'maria_db', 'first_name': 'Maria', 'last_name': 'DB', 'married': 1, 'created_at': '2026-06-04 14:59:24.761222'}
{'id': 1, 'title': 'Mi primer post', 'content': 'Contenido actualizado sobre Cloud Architecture en AWS', 'user_id': 1, 'created_at': '2026-06-04 14:59:24.762133'}
{'id': 2, 'title': 'SQLite Tips', 'content': 'Usando llaves foráneas eficientemente', 'user_id': 2, 'created_at': '2026-06-04 14:59:24.763243'}
{'id': 1, 'content': 'Excelente post! Muy recomendado.', 'user_id': 2, 'post_id': 1, 'created_at': '2026-06-04 14:59:24.764293'}
{'id': 2, 'content': 'Me sirvió mucho, gracias.', 'user_id': 1, 'post_id': 2, 'created_at': '2026-06-04 14:59:24.765346'}


In [12]:
def delete_record_raw(table_name: str, record_id: int):
    table_name = table_name.lower()
    if table_name not in VALID_TABLES:
        raise ValueError(f"Tabla inválida: {table_name}. Permisibles: {VALID_TABLES}")
    query = f"DELETE FROM {table_name} WHERE id = :id"
    
    with Session(engine) as session:
        try:
            session.execute(text(query), {"id": record_id})
            session.commit()
            print(f"🗑️ Record {record_id} successfully deleted from {table_name}.")
            return True
        except IntegrityError as e:
            session.rollback()
            print(f"❌ Error de integridad referencial: no se puede eliminar el registro {record_id} de {table_name} porque tiene dependencias.\nDetalle: {e.orig}")

In [20]:
delete_record_raw(table_name="comment", record_id=2)
delete_record_raw(table_name="user", record_id=2)

🗑️ Record 2 successfully deleted from comment.
🗑️ Record 2 successfully deleted from user.


True

In [13]:
read_all_raw("user")
read_all_raw("post")
read_all_raw("comment")

{'id': 1, 'username': 'jorge_laco', 'first_name': 'Jorge', 'last_name': 'Laco', 'married': 1, 'created_at': '2026-06-04 14:59:24.759962'}
{'id': 2, 'username': 'maria_db', 'first_name': 'Maria', 'last_name': 'DB', 'married': 1, 'created_at': '2026-06-04 14:59:24.761222'}
{'id': 1, 'title': 'Mi primer post', 'content': 'Contenido actualizado sobre Cloud Architecture en AWS', 'user_id': 1, 'created_at': '2026-06-04 14:59:24.762133'}
{'id': 2, 'title': 'SQLite Tips', 'content': 'Usando llaves foráneas eficientemente', 'user_id': 2, 'created_at': '2026-06-04 14:59:24.763243'}
{'id': 1, 'content': 'Excelente post! Muy recomendado.', 'user_id': 2, 'post_id': 1, 'created_at': '2026-06-04 14:59:24.764293'}
{'id': 2, 'content': 'Me sirvió mucho, gracias.', 'user_id': 1, 'post_id': 2, 'created_at': '2026-06-04 14:59:24.765346'}


In [14]:
def unified_query_sqlmodel():
    statement = (
        select(
            User.id,
            User.username,
            User.married,
            Post.id,
            Post.title,
            Post.content,
            Comment.id,
            Comment.content,
            Comment.created_at,
        )
        .join(Post, User.id == Post.user_id, isouter=True)
        .join(Comment, Post.id == Comment.post_id, isouter=True)
    )

    with Session(engine) as session:
        result = session.execute(statement)

        for row in result:
            print(row)

In [21]:
unified_query_sqlmodel()

(1, 'jorge_laco', True, 1, 'Mi primer post', 'Contenido actualizado sobre Cloud Architecture en AWS', 1, 'Excelente post! Muy recomendado.', datetime.datetime(2026, 6, 4, 14, 59, 24, 764293))
(2, 'maria_db', True, 2, 'SQLite Tips', 'Usando llaves foráneas eficientemente', 2, 'Me sirvió mucho, gracias.', datetime.datetime(2026, 6, 4, 14, 59, 24, 765346))


In [22]:
def unified_query_raw():
    query = """
        SELECT 
            u.id AS user_id,
            u.username,
            u.married,
            p.id AS post_id,
            p.title AS post_title,
            p.content AS post_content,
            c.id AS comment_id,
            c.content AS comment_text,
            c.created_at AS comment_date
        FROM user u
        LEFT JOIN post p ON u.id = p.user_id
        LEFT JOIN comment c ON p.id = c.post_id;
    """
    
    with Session(engine) as session:
        result = session.execute(text(query))
        for row in result:
            print(row)

In [23]:
unified_query_raw()

(1, 'jorge_laco', 1, 1, 'Mi primer post', 'Contenido actualizado sobre Cloud Architecture en AWS', 1, 'Excelente post! Muy recomendado.', '2026-06-04 14:59:24.764293')
(2, 'maria_db', 1, 2, 'SQLite Tips', 'Usando llaves foráneas eficientemente', 2, 'Me sirvió mucho, gracias.', '2026-06-04 14:59:24.765346')


---

<a id='-mongo-shell'></a>
## 🔹 5: PARTE 5: MongoDB desde Mongo Shell

A diferencia de SQL, MongoDB es **NoSQL** y está **orientado a documentos (BSON)**. No tiene un esquema rígido (schema-less), lo que permite guardar objetos complejos con diferentes estructuras en la misma colección.

### Comandos de Mongo Shell:
- `use blog_db`: Crea o cambia de base de datos.
- `db.users.insertOne({name: 'Ana'})`: Crea colección y documento.
- `db.users.find({age: {$gt: 18}})`: Filtra documentos.
- `db.users.updateOne({name: 'Ana'}, {$set: {married: true}})`: Actualiza.
- **BSON:** Es el formato binario de JSON que usa Mongo para ser más rápido y soportar más tipos de datos.

---

<a id='-pymongo'></a>
## 🔹 6: PARTE 6: PyMongo en Python

PyMongo es el driver oficial de MongoDB. Es ideal para cuando necesitas control total sobre los diccionarios.

In [8]:
# !uv add pymongo

In [1]:
import pymongo

# Conexión (Asumiendo Mongo local)
try:
    client = pymongo.MongoClient("mongodb://localhost:27017/", serverSelectionTimeoutMS=2000)
    db = client["lab_database"]
    users_col = db["users"]
except Exception as e:
    print("⚠️ MongoDB no disponible localmente. Mostrando lógica de código solamente.")

In [2]:
# Insertar
marta_doc = {"username": "marta99", "first_name": "Marta", "tags": ["python", "mongo"]}
result_marta = users_col.insert_one(marta_doc)

jorge_doc = {"username": "jorge_laco", "first_name": "Jorge", "tags": ["sql", "architecture"]}
result_jorge = users_col.insert_one(jorge_doc)

In [3]:
# READ
for user in users_col.find():
    print(user)

marta_found = users_col.find_one({"username": "marta99"})

{'_id': ObjectId('6a223b5232e23226b70ed48b'), 'username': 'marta99', 'first_name': 'Marta', 'tags': ['python', 'mongo']}
{'_id': ObjectId('6a223b5232e23226b70ed48c'), 'username': 'jorge_laco', 'first_name': 'Jorge', 'tags': ['sql', 'architecture']}


In [4]:
# UPDATE
update_result = users_col.update_one(
    {"username": "marta99"},                                # Filtro (El "WHERE")
    {"$set": {"tags": ["python", "mongo", "fastapi"]}}      # La actualización
)

In [5]:
# DELETE
delete_result = users_col.delete_one({"username": "jorge_laco"})

### 🔹 Aggregation Pipeline en MongoDB

MongoDB tiene un poderoso sistema de **aggregation pipeline** (`$match`, `$group`, `$lookup`, `$sort`) que permite procesar datos directamente en el motor de base de datos, similar a GROUP BY y JOINs en SQL.

In [ ]:
# Aggregation pipeline: contar cuantos posts tiene cada usuario
pipeline = [
    {"$group": {"_id": "$author", "total_posts": {"$sum": 1}}},
    {"$sort": {"total_posts": -1}}
]
# Asumiendo que tienes una colección 'posts'
# results = db.posts.aggregate(pipeline)
# for r in results:
#     print(r)

---

<a id='-mongoengine'></a>
## 🔹 7: PARTE 7: MongoEngine (ODM)

**MongoEngine** es un ODM (Object-Document Mapper). Nos permite definir clases para nuestros documentos de forma similar a como lo hicimos en SQLModel.

In [6]:
# !uv add mongoengine

In [7]:
from datetime import datetime
from mongoengine import Document, StringField, BooleanField, ReferenceField, ListField, connect, DateTimeField, CASCADE

connect(db="test_db", host="localhost", port=27017)

# 1. Definición de Modelos
class MUser(Document):
    username = StringField(required=True, unique=True)
    first_name = StringField(required=True)
    last_name = StringField(required=True)
    married = BooleanField(default=False)
    tags = ListField(StringField(), default=[])
    created_at = DateTimeField(default=datetime.utcnow)
    
    # NOTA SOBRE RELACIONES INVERSAS:
    # A diferencia de SQLModel (donde pones posts: list["Post"]), en MongoDB 
    # es una mala práctica guardar arrays infinitos de referencias en el padre.
    # En su lugar, hacemos la consulta directamente desde el hijo.

class MPost(Document):
    title = StringField(required=True)
    content = StringField(required=True)
    author = ReferenceField(MUser, reverse_delete_rule=CASCADE, required=True)
    created_at = DateTimeField(default=datetime.utcnow)

class MComment(Document):
    content = StringField(required=True)
    author = ReferenceField(MUser, reverse_delete_rule=CASCADE, required=True)
    post = ReferenceField(MPost, reverse_delete_rule=CASCADE, required=True)
    created_at = DateTimeField(default=datetime.utcnow)

# 2. CRUD Operations
def create_user(username, first_name, last_name, married):
    user = MUser(
        username=username, 
        first_name=first_name, 
        last_name=last_name, 
        married=married
    )
    user.save()
    return user

def create_post(author_username, title, content):
    author = MUser.objects(username=author_username).first()
    post = MPost(title=title, content=content, author=author)
    post.save()
    return post

def read_user_and_posts(username: str):
    user = MUser.objects(username=username).first()
    user_posts = MPost.objects(author=user)
    print(f"{user.first_name} {user.last_name} (Casado: {user.married})")
    for post in user_posts:
        print(f"   - {post.title} (Creado: {post.created_at.strftime('%Y-%m-%d')})")

def update_user_status(username: str, is_married: bool):
    user = MUser.objects(username=username).first()
    user.married = is_married
    user.save()

def delete_user(username: str):
    user = MUser.objects(username=username).first()
    if user:
        try:
            user.delete()
            print(f"🗑️ Usuario '{username}' eliminado correctamente.")
        except Exception as e:
            print(f"❌ Error al eliminar '{username}': {e}")
    else:
        print(f"❌ Usuario '{username}' no encontrado.")

In [8]:
create_user("jorge_laco", "Jorge", "Laco", married=False)
create_user("maria_db", "Maria", "DB", married=True)

<MUser: MUser object>

In [9]:
create_post("jorge_laco", "Arquitectura Cloud", "AWS y MongoDB son geniales.")
create_post("jorge_laco", "Python Tips", "Usa MongoEngine para facilitar tu vida.")
create_post("maria_db", "NoSQL vs SQL", "Depende del caso de uso.")

<MPost: MPost object>

In [10]:
read_user_and_posts("jorge_laco")

Jorge Laco (Casado: False)
   - Arquitectura Cloud (Creado: 2026-06-05)
   - Python Tips (Creado: 2026-06-05)


In [11]:
update_user_status("jorge_laco", is_married=True)

In [12]:
delete_user("jorge_laco")

---

<a id='-comparativa'></a>
## 🔹 8: PARTE 8: Comparativa: SQL vs NoSQL

Para cerrar, analicemos cuándo usar cada una:

| Característica | SQLite (Relacional) | MongoDB (Documental) |
| :--- | :--- | :--- |
| **Esquema** | Rígido (Tablas fijas) | Flexible (Sin esquema) |
| **Relaciones** | JOINs potentes | Referencias o Embebido |
| **Integridad** | Alta (Constraints/FK) | Responsabilidad del programador |
| **Escalabilidad**| Vertical (Mejor CPU/RAM) | Horizontal (Más servidores) |
| **Caso de Uso** | Finanzas, ERPs, Apps Locales | Redes Sociales, Big Data, CMS |

### 🧪 Mini-Actividad: El mismo problema en ambos mundos

Imagina que tienes un sistema de **biblioteca** con libros, autores y préstamos:

| Aspecto | Enfoque SQL (SQLModel) | Enfoque NoSQL (MongoDB) |
| :--- | :--- | :--- |
| **Modelado** | `Author` (1) → `Book` (N) → `Loan` (N) con FK | Documento `Book` embebe `author`, `Loan` como colección separada |
| **Consulta** | `SELECT * FROM book JOIN author ON ...` | `db.books.find({author.name: "..."})` |
| **Integridad** | Garantizada por la BD | Validación en la app |

💬 **Discusión en clase**: ¿Qué enfoque elegirías para cada escenario? ¿Por qué?

### Conclusión
No existe una "mejor" base de datos, sino la herramienta adecuada para el problema adecuado. **SQLite** es imbatible para aplicaciones locales y datos estructurados, mientras que **MongoDB** brilla cuando la rapidez de desarrollo y la flexibilidad de los datos son la prioridad.

**Regla de oro:** Si tus datos tienen relaciones claras y necesitas integridad → SQL. Si tus datos son variados, crecen rápido o cambian de forma constante → NoSQL.